<a href="https://colab.research.google.com/github/Nirmalkumarganesan95/SQL-Data-Engineering-/blob/main/SQL_Part_I.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Day 1: INNER JOIN (Most Important SQL Concept)

**INNER JOIN Definition**

INNER JOIN returns only the rows where the matching value exists in both tables.

Think of it as the common (overlapping) records between two tables.

Syntax
```
SELECT column_names
FROM table1
INNER JOIN table2
ON table1.column = table2.column;
```



Visual Representation
```
Customers             Orders

1 ✔                   1 ✔
2 ✔                   2 ✔
3 ✘                   4 ✘

INNER JOIN returns only

1
2
```



In [29]:
import sqlite3

In [68]:
connection = sqlite3.connect(':memory:')
cursor = connection.cursor()

In [69]:
cursor.execute("""
    CREATE TABLE IF NOT EXISTS Customers (
    customer_id INT,
    customer_name VARCHAR(50)
);""")

cursor.executemany("""
INSERT INTO Customers VALUES (?, ?)
""", [
    (1,'John'),
    (2,'David'),
    (3,'Emma')
])

connection.commit()

In [70]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS Orders (
  order_id INT,
  customer_id INT,
  amount INT
);""")

cursor.executemany("""
INSERT INTO Orders VALUES (?,?,?) """,
[
  (101,1,500),
  (102,2,900),
  (103,1,700),
  (104,4,300)
])

connection.commit()

In [71]:
cursor.execute(
    """
    SELECT
      c.customer_name,
      o.order_id,
      o.amount
    FROM Customers c
    INNER JOIN Orders o
    ON c.customer_id = o.customer_id;
""")

for row in cursor.fetchall():
  print(row)

('John', 101, 500)
('John', 103, 700)
('David', 102, 900)


In [72]:
#Exercise 2
query = """
  SELECT c.customer_name,
    o.amount
  FROM Customers c
  INNER JOIN Orders o
  ON c.customer_id = o.customer_id;
 """

cursor.execute(query)

for row in cursor.fetchall():
  print(row)

('John', 500)
('John', 700)
('David', 900)


In [73]:
#Exercise 3 Find all customers who have placed an order.

query = """
  SELECT DISTINCT c.customer_name
  FROM Customers c
  INNER JOIN Orders o
  ON c.customer_id = o.customer_id;
"""
cursor.execute(query)

for row in cursor.fetchall():
  print(row)

('John',)
('David',)


In [74]:
#Exercise 4
cursor.execute("""
    CREATE TABLE IF NOT EXISTS Products (
    product_id INT,
    product_name VARCHAR(50)
);""")

cursor.executemany("""
INSERT INTO Products VALUES (?, ?)
""", [
    (1,'Laptop'),
    (2,'Mouse'),
    (3,'Keyboard')
])

connection.commit()

cursor.execute("""
    CREATE TABLE IF NOT EXISTS Sales (
    sale_id INT,
    product_id INT,
    quantity INT
);""")

cursor.executemany("""
INSERT INTO Sales VALUES (?, ?, ?)
""", [
    (1, 1, 2),
    (2, 2, 5),
    (3, 4, 1)
])

connection.commit()

In [75]:
# Write an INNER JOIN to display: Product name Quantity
query = """
  SELECT p.product_name, s.quantity
  FROM Products p
  INNER JOIN Sales s
  ON p.product_id = s.product_id;
 """
cursor.execute(query)

for row in cursor.fetchall():
  print(row)

('Laptop', 2)
('Mouse', 5)


In [76]:
#Exercise 5
cursor.execute("""
    CREATE TABLE IF NOT EXISTS Employees (
    employee_id INT,
    employee_name VARCHAR(50),
    department_id INT
);""")

cursor.executemany("""
INSERT INTO Employees VALUES (?, ?, ?)
""", [
    (1,'Alice', 101),
    (2,'Bob', 102),
    (3,'Charlie', 101),
    (4,'David', 103)
])

connection.commit()

cursor.execute("""
    CREATE TABLE IF NOT EXISTS Departments (
    department_id INT,
    department_name VARCHAR(50)
);""")

cursor.executemany("""
INSERT INTO Departments VALUES (?, ?)
""", [
    (101, 'HR'),
    (102, "IT"),
    (104, "Finance")
])

connection.commit()

In [77]:
#Task: Write a query to display: Employee name, Department name

query = """
  SELECT e.employee_name, d.department_name
  FROM Employees e
  INNER JOIN Departments d
  ON e.department_id = d.department_id;
"""

cursor.execute(query)

for i in cursor.fetchall():
  print(i)

('Alice', 'HR')
('Bob', 'IT')
('Charlie', 'HR')


In [78]:
#Exercise 5
cursor.execute("""
    CREATE TABLE IF NOT EXISTS Departments_1 (
    department_id INT,
    department_name VARCHAR(50)
);""")

cursor.executemany("""
INSERT INTO Departments_1 VALUES (?, ?)
""", [
    (101, "HR"),
    (101, "Human Resources"),
    (102, "IT")
])

connection.commit()


In [79]:
query = """
  SELECT department_id, COUNT(*)
  FROM Departments_1
  GROUP BY department_id
  HAVING COUNT(*) > 1;"""

cursor.execute(query)

for i in cursor.fetchall():
  print(i)

(101, 2)


#Day 2: LEFT JOIN (Very Important for Interviews & Data Engineering)



```
Why Do We Need LEFT JOIN?

Imagine your manager asks:

   "Show all customers, even if they haven't placed any orders."

INNER JOIN cannot do this because it removes non-matching rows.

That's why we use LEFT JOIN.
```



**LEFT JOIN Definition**

```
A LEFT JOIN returns:

✅ All rows from the left table.
✅ Matching rows from the right table.
✅ If no match exists, SQL returns NULL for the right table's columns.
```



**Syntax**

```
SELECT columns
FROM table1
LEFT JOIN table2
ON table1.column = table2.column;

Memory Trick:
LEFT JOIN keeps everything on the left.
```



**Visual Representation**
```
Customers (LEFT)

John  --------> Order ✔
David --------> Order ✔
Emma  --------> No Order ✘

LEFT JOIN returns

John
David
Emma
```



INNER JOIN vs LEFT JOIN

| INNER JOIN                           | LEFT JOIN                                            |
| ------------------------------------ | ---------------------------------------------------- |
| Returns only matching rows           | Returns all rows from the left table                 |
| Non-matching rows are removed        | Non-matching rows appear with `NULL`                 |
| Used when you only want related data | Used when you want complete data from the left table |


In [80]:
cursor.execute("DELETE FROM Customers")
cursor.execute("DELETE FROM Orders")

cursor.execute("""
    CREATE TABLE IF NOT EXISTS Customers (
    customer_id INT,
    customer_name VARCHAR(50)
);""")

cursor.executemany("""
INSERT INTO Customers VALUES (?, ?)
""", [
    (1,'John'),
    (2,'David'),
    (3,'Emma')
])

connection.commit()

cursor.execute("""
    CREATE TABLE IF NOT EXISTS Orders (
    order_id INT,
    customer_id INT,
    amount INT
);""")

cursor.executemany("""
INSERT INTO Orders VALUES (?, ?, ?)
""", [
    (101, 1, 500),
    (102, 2, 900)
])

connection.commit()

In [81]:
#Exercise 1
#LEFT JOIN
query = """
  select c.customer_name, o.order_id, o.amount
  from Customers c
  left join Orders o
  on c.customer_id = o.customer_id;
"""
cursor.execute(query)

for i in cursor.fetchall():
  print(i)

('John', 101, 500)
('David', 102, 900)
('Emma', None, None)


In [85]:
#Exercise 2 Customers Without Orders
query = """
select c.customer_name
from Customers c
left join Orders o
on c.customer_id = o.customer_id
where o.order_id is null;"""

cursor.execute(query)

for i in cursor.fetchall():
  print(i)

('Emma',)


In [86]:
#Exercise 3
cursor.execute("DELETE FROM Products")
cursor.execute("DELETE FROM Sales")

cursor.execute("""
    CREATE TABLE IF NOT EXISTS Products (
    product_id INT,
    product_name VARCHAR(50)
);""")

cursor.executemany("""
INSERT INTO Products VALUES (?, ?)
""", [
    (1,'Laptop'),
    (2,'Mouse'),
    (3,'Keyboard'),
    (4, "Monitor")
])

connection.commit()

cursor.execute("""
    CREATE TABLE IF NOT EXISTS Sales (
    sale_id INT,
    product_id INT,
    quantity INT
);""")

cursor.executemany("""
INSERT INTO Sales VALUES (?, ?, ?)
""", [
    (1, 1, 2),
    (2, 2, 5),
    (3, 1, 1)
])

connection.commit()

In [87]:
query = """
select p.product_name, s.quantity
from Products p
left join Sales s
on p.product_id = s.product_id
"""
cursor.execute(query)

for i in cursor.fetchall():
  print(i)

('Laptop', 1)
('Laptop', 2)
('Mouse', 5)
('Keyboard', None)
('Monitor', None)


In [88]:
#Exercise 4 find products that have never been sold.
query = """
select p.product_name
from Products p
left join Sales s
on p.product_id = s.product_id
where s.quantity is null; """

cursor.execute(query)

for i in cursor.fetchall():
  print(i)

('Keyboard',)
('Monitor',)


In [91]:
#Challenge Question (Interview Level)
cursor.execute("DELETE FROM Employees")
cursor.execute("DELETE FROM Departments")

cursor.execute("""
    CREATE TABLE IF NOT EXISTS Employees (
    employee_id INT,
    employee_name VARCHAR(50),
    department_id INT
);""")

cursor.executemany("""
INSERT INTO Employees VALUES (?, ?, ?)
""", [
    (1,'Alice', 101),
    (2,'Bob', 102),
    (3,'Charlie', 101),
    (4,'David', "NULL")

])

connection.commit()

cursor.execute("""
    CREATE TABLE IF NOT EXISTS Departments (
    department_id INT,
    department_name VARCHAR(50)
);""")

cursor.executemany("""
INSERT INTO Departments VALUES (?, ?)
""", [
    (101, 'HR'),
    (102, "IT"),
])

connection.commit()

In [92]:
query = """
select e.employee_name, d.department_name
from Employees e
left join Departments d
on e.department_id = d.department_id;
 """

cursor.execute(query)

for i in cursor.fetchall():
  print(i)

('Alice', 'HR')
('Bob', 'IT')
('Charlie', 'HR')
('David', None)


#Day 3: RIGHT JOIN



```
What is RIGHT JOIN?

A RIGHT JOIN returns:

✅ All rows from the right table
✅ Matching rows from the left table
✅ If no match exists, columns from the left table become NULL
```





```
Memory Trick
LEFT JOIN
Keeps everything on the LEFT table.

RIGHT JOIN
Keeps everything on the RIGHT table.
```



Syntax
```
SELECT columns
FROM table1
RIGHT JOIN table2
ON table1.column = table2.column;
```



In [97]:
cursor.execute("DELETE FROM Customers")
cursor.execute("DELETE FROM Orders")

cursor.execute("""
    CREATE TABLE IF NOT EXISTS Customers (
    customer_id INT,
    customer_name VARCHAR(50)
);""")

cursor.executemany("""
INSERT INTO Customers VALUES (?, ?)
""", [
    (1,'John'),
    (2,'David'),
    (3,'Emma')
])

connection.commit()

cursor.execute("""
    CREATE TABLE IF NOT EXISTS Orders (
    order_id INT,
    customer_id INT,
    amount INT
);""")

cursor.executemany("""
INSERT INTO Orders VALUES (?, ?, ?)
""", [
    (101, 1, 500),
    (102, 2, 900),
    (103, 4, 600)
])

connection.commit()

In [98]:
query = """
  select c.customer_name, o.order_id, o.amount
  from Customers c
  right join Orders o
  on c.customer_id = o.customer_id;
"""
cursor.execute(query)

for i in cursor.fetchall():
  print(i)

OperationalError: RIGHT and FULL OUTER JOINs are not currently supported

**Visual Representation**

```
Customers              Orders

John    ✔
David   ✔
Emma    ✘

                     101 ✔
                     102 ✔
                     103 ✔

RIGHT JOIN returns

John
David
NULL (Order 103)
```



#Day 4: FULL OUTER JOIN (FULL JOIN)



```
What is FULL OUTER JOIN?

A FULL OUTER JOIN returns:

✅ Matching rows
✅ Unmatched rows from the left table
✅ Unmatched rows from the right table

If there is no match, the missing columns are filled with NULL.
```



In [99]:
cursor.execute("DELETE FROM Customers")
cursor.execute("DELETE FROM Orders")

cursor.execute("""
    CREATE TABLE IF NOT EXISTS Customers (
    customer_id INT,
    customer_name VARCHAR(50)
);""")

cursor.executemany("""
INSERT INTO Customers VALUES (?, ?)
""", [
    (1,'John'),
    (2,'David'),
    (3,'Emma')
])

connection.commit()

cursor.execute("""
    CREATE TABLE IF NOT EXISTS Orders (
    order_id INT,
    customer_id INT,
    amount INT
);""")

cursor.executemany("""
INSERT INTO Orders VALUES (?, ?, ?)
""", [
    (101, 1, 500),
    (102, 2, 900),
    (103, 4, 600)
])

connection.commit()

In [101]:
query = """
  select c.customer_name, o.order_id, o.amount
  from Customers c
  FULL OUTER JOIN Orders o
  on c.customer_id = o.customer_id;
"""
cursor.execute(query)

for i in cursor.fetchall():
  print(i)

OperationalError: RIGHT and FULL OUTER JOINs are not currently supported

In [103]:
#FULL OUTER JOIN alternative
query = """
SELECT c.customer_name, o.order_id, o.amount
FROM Customers c
LEFT JOIN Orders o
ON c.customer_id = o.customer_id

UNION

SELECT c.customer_name, o.order_id, o.amount
FROM Orders o
LEFT JOIN Customers c
ON c.customer_id = o.customer_id;

"""
cursor.execute(query)

for i in cursor.fetchall():
  print(i)

(None, 103, 600)
('David', 102, 900)
('Emma', None, None)
('John', 101, 500)


Visual Representation

```
Visual Representation
Customers                Orders

John     ✔               101 ✔
David    ✔               102 ✔
Emma     ✘               103 ✘

FULL JOIN returns

John
David
Emma
Order 103
```



#5: SELF JOIN (Very Important for Interviews)